# Conditional Independence

In our review of the fundamentals of probability, we reminded ourselves of the notion of *independence*. If a random variable is independent, then the univariate distribution can be factored out of the joint distribution. For example, given $P(X_1, X_2, X_3)$, then if $X_2$ is independent of $X_1$ and $X_3$, the joint probability distribution can be written as

$$
P(X_1,X_2,X_3) = P(X_2)P(X_1,X_3)
$$

Independence here is *unconditional*: the distribution of $X_2$ is always the same and does not depend on the values of $X_1$ and $X_3$. We write this as $X_2\perp\!\!\!\perp (X_1,X_3)$. Examples of independence might include:

* Being good at mathematics is independent of your eye colour.
* Being good at mathematics is not independent of being good at programming, but eye colour is independent of both.
* Height and weight are not independent, but hair colour is independent of both.

In each of these examples, the distribution of the independent variable does not depend on the values of the other variable(s). This is an extremely limiting condition that will often not be satisfied in a real-world problem. (Even the "naive" assumption made by naive Bayes classifiers is weaker: the features are assumed to be independent *given the class*, which is a conditional independence.)

## Conditional independence

What if two variables are in general not independent, but can become independent under certain conditions? For example, ice-cream sales and incidents of sunburn are not, in general, independent: they both tend to be high in the same months. However, if we *condition* on a third variable such as the level of sunshine, then these quantities become (approximately) independent: on days with the same amount of sunshine, knowing the ice-cream sales tells us nothing more about the number of sunburn cases. This is an example of **conditional independence**.

Variables $X_1$ and $X_2$ are conditionally independent given $X_3$ when they become independent once the value of $X_3$ is known. We write this as $X_1\perp\!\!\!\perp X_2\mid X_3$, and express it (in the simple three-variable case) as

$$
P(X_1,X_2\vert X_3) = P(X_1\vert X_3)P(X_2\vert X_3)
$$

There are two other, equivalent, ways of expressing this. The first is

$$P(X_1\vert X_2,X_3)=P(X_1\vert X_3)$$

and the second, which we get by multiplying the first expression by $P(X_3)$ and using the product rule $P(X_1,X_2,X_3)=P(X_1,X_2\vert X_3)P(X_3)$, is

$$
P(X_1,X_2,X_3) = P(X_1\vert X_3)P(X_2\vert X_3)P(X_3)
$$

This last form is a *factorisation* of the joint distribution, and it is the idea on which Bayesian networks are built.

**Proof:** 

Assume $P(X_1,X_2\vert X_3)=P(X_1\vert X_3)P(X_2\vert X_3)$ and that $P(X_2,X_3)>0$. Using the definition of conditional probability, $P(A\vert B)=P(A,B)/P(B)$, to expand each term we obtain

$$
\frac{P(X_1,X_2,X_3)}{P(X_3)} = \frac{P(X_1,X_3)}{P(X_3)} \frac{P(X_2,X_3)}{P(X_3)}
$$

Multiplying through by $P(X_3)$ and then dividing by $P(X_2,X_3)$ we obtain

$$
\frac{P(X_1,X_2,X_3)}{P(X_2,X_3)} = \frac{P(X_1,X_3)}{P(X_3)}
$$

The left hand side is $P(X_1\vert X_2,X_3)$ and the right hand side is $P(X_1\vert X_3)$, so

$$
P(X_1\vert X_2,X_3) = P(X_1\vert X_3)
$$

(we can use a similar argument to show that $P(X_2\vert X_1,X_3) = P(X_2\vert X_3)$ is also implied). Every step can be reversed, so the three forms of conditional independence are equivalent.

This permits a slightly different interpretation of conditional independence: conditioning on $X_2$ does not affect the distribution of $X_1$ when it is already conditioned on $X_3$. Notice that the expression on the right hand side does not involve $X_2$ at all: it is not that $X_2$ is independent, it is that it does not affect the distribution of $X_1$ when conditioning on $X_3$.

Consider the following observation: taller children are better at reading. This correlation implies that reading ability and height are not independent of each other. Can you think of a reason why height and reading ability might not be independent? There is no obvious biological link between them. However, both height and reading ability are correlated with something else: *age*. Taller children are better at reading because taller children are older, and older children are better at reading, both of which have sound biological causes.

If we condition the joint distribution of height and reading ability on age, then the two distributions will be independent of each other: for 8 year old children, there is no correlation between height and reading ability. A variable like age, which is a common cause of the two variables, is called a **confounder**.

## A worked example: repeating a test

Let us return to the medical test of section 3.0, where $D$ is the disease (prior $P(D)=0.1$) and the test has $P(T=+\vert D)=0.8$ and $P(T=+\vert\lnot D)=0.1$. Suppose that the test is repeated, giving two results $T_1$ and $T_2$. Provided that the errors made by the test are random, once we know whether or not the person has the disease the second result does not depend on the first:

$$P(T_1,T_2\vert D)=P(T_1\vert D)P(T_2\vert D)$$

That is, $T_1\perp\!\!\!\perp T_2\mid D$. In the notation we use in code, $P(A,B\vert C)$ is `PAB_C`, and arrays are indexed in the same order as the name. The joint distribution is $P(D,T_1,T_2)=P(D)P(T_1\vert D)P(T_2\vert D)$.

In [1]:
import numpy as np

PD = np.array([0.9, 0.1])            # P(D):     index 0 = no disease, 1 = disease
PT_D = np.array([[0.9, 0.2],         # P(T | D): rows t = 0 (-), 1 (+);  columns d = 0, 1
                 [0.1, 0.8]])

# Joint distribution PDT1T2[d][t1][t2] = P(D) P(T1|D) P(T2|D)
PDT1T2 = np.einsum('d,td,sd->dts', PD, PT_D, PT_D)
print("P(D, T1=+, T2=+) =", round(PDT1T2[1, 1, 1], 4), "  P(not D, T1=+, T2=+) =", round(PDT1T2[0, 1, 1], 4))

P(D, T1=+, T2=+) = 0.064   P(not D, T1=+, T2=+) = 0.009


Is $T_1$ independent of $T_2$? We marginalise over $D$ to find $P(T_1,T_2)$ and compare it with the product $P(T_1)P(T_2)$:

In [2]:
PT1T2 = PDT1T2.sum(axis=0)           # sum over D:  P(T1, T2)
PT1 = PT1T2.sum(axis=1)              # P(T1)
PT2 = PT1T2.sum(axis=0)              # P(T2)

print("P(T1, T2):")
print(np.round(PT1T2, 4))
print("P(T1) P(T2):")
print(np.round(np.outer(PT1, PT2), 4))
print("Independent?", np.allclose(PT1T2, np.outer(PT1, PT2)))

# A positive first test makes a positive second test more likely
PT2_T1 = PT1T2 / PT1[:, None]        # P(T2 | T1)[t1][t2]
print("P(T2=+)        =", round(PT2[1], 4))
print("P(T2=+ | T1=+) =", round(PT2_T1[1, 1], 4))

P(T1, T2):
[[0.733 0.097]
 [0.097 0.073]]
P(T1) P(T2):
[[0.6889 0.1411]
 [0.1411 0.0289]]
Independent? False
P(T2=+)        = 0.17
P(T2=+ | T1=+) = 0.4294


The two results are *not* independent: $P(T_1=+,T_2=+)=0.073$ but $P(T_1=+)P(T_2=+)=0.17^2\approx0.029$. Learning that the first test was positive raises the probability of a positive second test from 0.17 to 0.43, because it makes it more likely that the person has the disease. Now condition on $D$:

In [3]:
# P(T1, T2 | D)[t1][t2][d]: divide the joint by P(D)
PT1T2_D = np.einsum('dts->tsd', PDT1T2) / PD[None, None, :]

# Compare with the product P(T1 | D) P(T2 | D)
product = np.einsum('td,sd->tsd', PT_D, PT_D)
print("P(T1,T2|D) = P(T1|D) P(T2|D)?", np.allclose(PT1T2_D, product))

# Equivalent form: P(T2 | T1, D) = P(T2 | D)
PT2_T1D = PT1T2_D / PT_D[:, None, :]    # divide by P(T1 | D) to get P(T2 | T1, D)[t1][t2][d]
print("P(T2=+ | T1=+, D)     =", round(PT2_T1D[1, 1, 1], 4))
print("P(T2=+ | T1=-, D)     =", round(PT2_T1D[0, 1, 1], 4))
print("P(T2=+ | D)           =", round(PT_D[1, 1], 4))

P(T1,T2|D) = P(T1|D) P(T2|D)? True
P(T2=+ | T1=+, D)     = 0.8
P(T2=+ | T1=-, D)     = 0.8
P(T2=+ | D)           = 0.8


Once the disease status is known, the first result tells us nothing about the second: $T_1$ and $T_2$ are conditionally independent given $D$ even though they are not independent.

The code above can be generalised. Given any joint distribution of three variables, stored as `PXYZ[x][y][z]`, the following function tests whether $X\perp\!\!\!\perp Y\mid Z$ by comparing $P(X,Y\vert Z)$ with $P(X\vert Z)P(Y\vert Z)$.

In [ ]:
def cond_independent(PXYZ, tol=1e-9):
    """Test whether X and Y are conditionally independent given Z, for a joint distribution PXYZ[x][y][z]."""
    PZ = PXYZ.sum(axis=(0, 1))
    PXY_Z = PXYZ / PZ[None, None, :]                  # P(X, Y | Z)
    PX_Z = PXY_Z.sum(axis=1)                          # P(X | Z)[x][z]
    PY_Z = PXY_Z.sum(axis=0)                          # P(Y | Z)[y][z]
    return np.allclose(PXY_Z, np.einsum('xz,yz->xyz', PX_Z, PY_Z), atol=tol) # Returns True if two arrays are element-wise equal within a tolerance.

def independent(PXY, tol=1e-9):
    """Test whether X and Y are independent, for a joint distribution PXY[x][y]."""
    return np.allclose(PXY, np.outer(PXY.sum(axis=1), PXY.sum(axis=0)), atol=tol)

# Apply to the repeated test, treating (T1, T2, D) as (X, Y, Z)
PT1T2D = np.einsum('dts->tsd', PDT1T2)
print("T1 independent of T2?          ", independent(PT1T2))
print("T1 independent of T2, given D? ", cond_independent(PT1T2D))

T1 independent of T2?           False
T1 independent of T2, given D?  True


## Conditional independence reduces the number of parameters

Conditional independence is the key to the factorisations that make Bayesian networks efficient. Suppose that we have a disease $D$ and $n$ different binary test results $T_1,\dots,T_n$, each of which is conditionally independent of the others given $D$. The full joint distribution of $n+1$ binary variables has $2^{n+1}-1$ free parameters, but the factorisation

$$P(D,T_1,\dots,T_n)=P(D)\prod_{i=1}^n P(T_i\vert D)$$

needs only one parameter for $P(D)$ and two for each $P(T_i\vert D)$ (one for each value of $D$), a total of $2n+1$. This is the structure assumed by *naive Bayes* classifiers.

In [5]:
print(" n   full joint        factorised")
for n in (1, 2, 5, 10, 20, 30):
    print(f"{n:2d}   {2**(n+1) - 1:>14,}   {2*n + 1:>8}")

 n   full joint        factorised
 1                3          3
 2                7          5
 5               63         11
10            2,047         21
20        2,097,151         41
30    2,147,483,647         61


## Three structures

Conditional independence arises in three basic patterns, each involving a "middle" variable $Z$ and two other variables $X$ and $Y$. We draw them with arrows from cause to effect, as we will do for Bayesian networks. They are the building blocks from which all Bayesian networks are made, and it is worth learning to recognise them.

| Structure | Factorisation | $X$ and $Y$ independent? | Given $Z$? | Example |
|:--|:--|:--|:--|:--|
| Chain $X\to Z\to Y$ | $P(X)P(Z\vert X)P(Y\vert Z)$ | No | Yes | Smoking $\to$ tar in the lungs $\to$ cancer |
| Fork $X\leftarrow Z\to Y$ | $P(Z)P(X\vert Z)P(Y\vert Z)$ | No | Yes | Height $\leftarrow$ age $\to$ reading ability |
| Collider $X\to Z\leftarrow Y$ | $P(X)P(Y)P(Z\vert X,Y)$ | **Yes** | **No** | Burglary $\to$ alarm $\leftarrow$ earthquake |

In a *chain*, $X$ influences $Y$ only through $Z$ ($Z$ is a *mediator*). In a *fork*, $Z$ is a common cause (a confounder) of $X$ and $Y$, as in the height and reading example. In both cases $X$ and $Y$ are dependent, but become independent when we observe $Z$.

The *collider* works the other way round. Two independent causes $X$ and $Y$ have a common effect $Z$. They are independent *until we observe the effect*, at which point they become dependent. This is the opposite of what happens in the chain and the fork, and it is perhaps the least intuitive of the three, so we look at it in detail.

### Explaining away

A house has an alarm that can be triggered by a burglary $B$ or by an earthquake $E$. Burglaries and earthquakes are unrelated, so they are independent: $P(B,E)=P(B)P(E)$. The alarm $A$ depends on both, so $P(A,B,E)=P(B)P(E)P(A\vert B,E)$. Suppose that

* $P(B)=0.01$ and $P(E)=0.02$;
* $P(A\vert\lnot B,\lnot E)=0.001$, $P(A\vert\lnot B,E)=0.29$, $P(A\vert B,\lnot E)=0.94$ and $P(A\vert B,E)=0.95$.

In [6]:
PB = np.array([0.99, 0.01])
PE = np.array([0.98, 0.02])

# PA_BE[a][b][e] = P(A=a | B=b, E=e);  P(A=1|b,e) is given in the order (b,e) = (0,0), (0,1), (1,0), (1,1)
PA1_BE = np.array([[0.001, 0.29],
                   [0.94,  0.95]])
PA_BE = np.array([1 - PA1_BE, PA1_BE])

# Joint distribution PBEA[b][e][a] = P(B) P(E) P(A | B, E)
PBEA = np.einsum('b,e,abe->bea', PB, PE, PA_BE)

# B and E are independent
PBE = PBEA.sum(axis=2)
print("B and E independent?", independent(PBE))

B and E independent? True


Now we *observe* that the alarm is ringing, $A=1$. What is the probability of a burglary?

In [7]:
PA = PBEA.sum(axis=(0, 1))                 # P(A)
PBE_A = PBEA[:, :, 1] / PA[1]              # P(B, E | A=1)[b][e]
PB_A = PBE_A.sum(axis=1)                   # P(B | A=1)
PE_A = PBE_A.sum(axis=0)                   # P(E | A=1)

print(f"P(B=1)            = {PB[1]:.4f}")
print(f"P(B=1 | A=1)      = {PB_A[1]:.4f}")

P(B=1)            = 0.0100
P(B=1 | A=1)      = 0.5835


The alarm raises our belief in a burglary from 1% to 58%. Now we are also told that there has been an earthquake, $E=1$:

In [8]:
# P(B | A=1, E=e): restrict P(B, E | A=1) to column e and renormalise
PB_AE = PBE_A / PBE_A.sum(axis=0)[None, :]      # PB_AE[b][e]
print(f"P(B=1 | A=1, E=1) = {PB_AE[1, 1]:.4f}")
print(f"P(B=1 | A=1, E=0) = {PB_AE[1, 0]:.4f}")

# Given the alarm, B and E are no longer independent
print("B and E independent given A=1?", independent(PBE_A))
print("P(B, E | A=1):")
print(np.round(PBE_A, 4))
print("P(B | A=1) P(E | A=1):")
print(np.round(np.outer(PB_A, PE_A), 4))

P(B=1 | A=1, E=1) = 0.0320
P(B=1 | A=1, E=0) = 0.9047
B and E independent given A=1? False
P(B, E | A=1):
[[0.0602 0.3563]
 [0.5717 0.0118]]
P(B | A=1) P(E | A=1):
[[0.2632 0.1533]
 [0.3687 0.2148]]


Learning about the earthquake lowers the probability of a burglary from 58% to just 3%: the earthquake *explains away* the alarm, so there is much less need for the burglary to explain it. Conversely, if there was no earthquake the burglary becomes the only plausible explanation (90%). Although $B$ and $E$ are independent, once the alarm has been observed they are not: $B\not\perp\!\!\!\perp E\mid A$.

This also shows that neither independence nor conditional independence implies the other. In the fork and the chain $X$ and $Y$ are dependent but conditionally independent given $Z$; in the collider they are independent but become dependent given $Z$. Whether conditioning on a variable *removes* or *creates* a dependence depends on the structure, which is why we draw the graph.

A final caution. Conditional independence is a statement about probabilities. A confounder explains why two variables are correlated, but it does not by itself tell us that either causes the other. In Bayesian networks we will use the direction of the arrows to encode our beliefs about causes, but the independence relations that the network implies are what matter for calculation.